In [1]:
import jax
import jax.numpy as jnp

from typing import Callable

import equinox as eqx


# Define a RK solver

In [2]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_rk2(f, states, ST_top, ST_bot, dt, *args):
    
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST1 = ST_top
    states1 = states
    r1 = f(states1, ST1, *args)
    
    ST2 = ST_bot
    # states2 = jax.nn.relu(states+1.*r1*dt)
    states2 = states+1.*r1*dt
    r2 = f(states2, ST2, *args)

    states_new = states + (r1 + r2)/2. * dt
    # states_new = jax.nn.relu(states + (r1 + r2)/2. * dt)

    return states_new


# Define a general solver to solve TTDs-like function

Consider a variable $s$ varying on two dimensions, i.e., time $t$ and age $\tau$, represented as $s^j_i$ with $i$ and $j$ referring to the time and age indices, respectively. We can represent the evolution of $s^j_i$ through the following ODE system such that

\begin{align}
\frac{ds_i}{dt} = f_s(s_i, S_i, \vec{w}), 
\end{align}

where $\vec{w}$ is the parameter vector and $S_i = \int^{t_j-T_i+\Delta t}_0 s(\tau, t_j) d\tau = \sum^j_{k=i+1} s^j_i \Delta t$. The initial conditions, i.e., $s^0_k$, are prescribed.

We are interested at solving $\frac{\partial s^j_i}{\partial \vec{w}}$. Based on implicit function theorem, we can get the following

\begin{align}
\frac{d}{dt} \frac{\partial s^j_i}{\partial \vec{w}} &= \frac{d f_s}{d \vec{w}} \\
  &= \frac{\partial f_s}{\partial s_i}\frac{\partial s_i}{\partial \vec{w}} + \frac{\partial f_s}{\partial S_i}\frac{\partial S_i}{\partial \vec{w}} + \frac{\partial f_s}{\partial \vec{w}},
\end{align}

where $\frac{\partial S_i}{\partial \vec{w}} = \sum^j_{k=i+1} \frac{\partial s_i}{\partial \vec{w}} \Delta t$.

In [3]:
def f(sT, ST, a, b):
    return a * (b - sT) + ST
    # return a * (b - sT) + a*ST

fvec = jax.vmap(f, in_axes=(0,0,None,None))


In [4]:
def solve_ttd(
    sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args, solver, f
):
    # jax.debug.print('args: {x}', x=args)
    
    def step_τ(states, x):
        # sT_init_τ : (1,,)
        sT_init_τ = x
    
        # sT, ST_top, ST_bot : (nt,)
        sT, ST_top, ST_bot = states
    
        # Solve the ODE system
        sT_new = solver(
            f, sT, ST_top[:-1], ST_bot[1:], dt, *args
        )

        # Update ST
        ST_top_new = ST_bot
        ST_bot_new = jnp.concat([sT_init_τ[None] * dt, ST_bot[1:] + sT_new * dt])
        # ST_top_new = ST_top
        # ST_bot_new = ST_bot
        # jax.debug.print('sT_new: {x}; ST_bot: {y}; args: {z}', x=sT_new, y=ST_bot, z=args)
        
        # Variables as the initial condition to the next step
        sT_new_rotate = jnp.concat([sT_init_τ[None],sT_new[:-1]])
        
        return (sT_new_rotate, ST_top_new, ST_bot_new), sT_new
    
    _, sTs = jax.lax.scan(step_τ, (sT_0, ST_top_0, ST_bot_0), sT_init)

    return sTs


In [5]:
t0, nt, dt = 0, 10, 0.4

sT_init = jnp.array([10., 15., 20., 30., 50.])
# sT_init = jnp.array([0., 0., 0., 0., 0.])

# Initials at age = 0
sT_0 = jnp.zeros(nt)
ST_top_0, ST_bot_0 = jnp.zeros(nt+1), jnp.zeros(nt+1) 

# Arguments
a, b = 0.1, 0.2
args = [a, b]


2025-01-17 13:09:34.170119: W external/xla/xla/service/gpu/nvptx_compiler.cc:836] The NVIDIA driver's CUDA version is 12.4 which is older than the PTX compiler version (12.6.20). Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [6]:
sTs = solve_ttd(sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args, solver=solve_step_rk2, f=fvec)
sTs.shape, sTs

((5, 10),
 Array([[7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03],
        [9.6164675e+00, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02],
        [1.5957785e+01, 9.2498512e+00, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02],
        [2.2422407e+01, 1.6821121e+01, 8.9008932e+00, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02, 3.8349532e-02, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02],
        [3.4208199e+01, 2.5603579e+01, 1.7597830e+01, 8.5706577e+00,
         5.5525493e-02, 5.5525493e-02, 5.5525493e-02, 5.5525493e-02,
         5.5525493e-02, 5.5525493e-02]], dtype=float32))

## Differentiation without IFT

In [7]:
# # Without the self-defined JVP rule
# def evolve(sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args):
#     return solve_ttd(solve_step_rk2, fvec, sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args)

In [8]:
delta_sT_0 = jnp.zeros(nt)
delta_sT_init = jnp.zeros(sT_init.shape)
delta_ST_top_0 = jnp.zeros(ST_top_0.size)
delta_ST_bot_0 = jnp.zeros(ST_bot_0.shape)

delta_dt = 0.

delta_a = 1.
delta_b = 0.
delta_args = [delta_a, delta_a]


In [9]:
primals = (sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args)
tangents = ((delta_sT_0, delta_ST_top_0, delta_ST_bot_0, delta_sT_init, delta_dt, *delta_args))
r1, r2 = eqx.filter_jvp(solve_ttd, primals, tangents, solver=solve_step_rk2, f=fvec)
# r1.shape, r2.shape
r1, r2

(Array([[7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03],
        [9.6164675e+00, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02],
        [1.5957785e+01, 9.2498512e+00, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02],
        [2.2422407e+01, 1.6821121e+01, 8.9008932e+00, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02, 3.8349532e-02, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02],
        [3.4208199e+01, 2.5603579e+01, 1.7597830e+01, 8.5706577e+00,
         5.5525493e-02, 5.5525493e-02, 5.5525493e-02, 5.5525493e-02,
         5.5525493e-02, 5.5525493e-02]], dtype=float32),
 Array([[  0.11600001,   0.11600001,   0.11600001,   0.11600001,
           0.11600001,   0.1

In [462]:
%timeit eqx.filter_jvp(solve_ttd, primals, tangents, solver=solve_step_rk2, f=fvec)

123 ms ± 3.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Differentiation with IFT

In [10]:
def solve_ttd_aug(
    aug_sT_0, aug_ST_top_0, aug_ST_bot_0, aug_sT_init, dt, *args, solver, f
):
    
    def step_τ(states, x):
        # aug_sT_init_τ : (2,)
        aug_sT_init_τ = x
    
        # aug_sT, aug_ST_top, aug_ST_bot : (2,nt), (2,nt+), (2,nt+1)
        aug_sT, aug_ST_top, aug_ST_bot = states
    
        # Solve the ODE system
        aug_sT_new = solver(
            f, aug_sT, aug_ST_top[...,:-1], aug_ST_bot[...,1:], dt, *args
        )

        # Update ST
        aug_ST_top_new = aug_ST_bot
        aug_ST_bot_new = jnp.concat([aug_sT_init_τ[...,None] * dt, aug_ST_bot[:,1:] + aug_sT_new * dt], axis=1)
        
        # Variables as the initial condition to the next step
        aug_sT_new_rotate = jnp.concat([aug_sT_init_τ[...,None],aug_sT_new[...,:-1]], axis=1)
        
        return (aug_sT_new_rotate, aug_ST_top_new, aug_ST_bot_new), aug_sT_new
    
    _, aug_sTs = jax.lax.scan(step_τ, (aug_sT_0, aug_ST_top_0, aug_ST_bot_0), aug_sT_init)

    return aug_sTs


In [11]:
# solve_ttd_jax = jax.custom_jvp(solve_ttd, nondiff_argnums=(0,1))
solve_ttd_jax = eqx.filter_custom_jvp(solve_ttd)

@solve_ttd_jax.def_jvp
def solve_ttd_jax_jvp(primals, tangents, *, solver, f):
    sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args = primals
    delta_sT_0, delta_ST_top_0, delta_ST_bot_0, delta_sT_init, delta_dt, *delta_args = tangents
    nargs = len(args)
    
    def f_aug(aug_sT, aug_ST, *args_and_delta_args):
        # print(args_and_delta_args)
        # print(aug_sT)
        # print(aug_ST)
        # primal_sT, tangent_sT = aug_sT[...,0], aug_sT[...,1]
        primal_sT, tangent_sT = aug_sT[0], aug_sT[1]
        primal_ST, tangent_ST = aug_ST[0], aug_ST[1]
        args, delta_args = args_and_delta_args[:nargs], args_and_delta_args[nargs:]
        # primal_dot, tangent_dot = jax.jvp(
        primal_dot, tangent_dot = eqx.filter_jvp(
            f, (primal_sT, primal_ST, *args), (tangent_sT, tangent_ST, *delta_args)
        )
        return jnp.stack([primal_dot, tangent_dot])

    aug_sT_0 = jnp.stack([sT_0, delta_sT_0])
    aug_ST_top_0 = jnp.stack([ST_top_0, delta_ST_top_0])
    aug_ST_bot_0 = jnp.stack([ST_bot_0, delta_ST_bot_0])
    aug_sT_init = jnp.stack([sT_init, delta_sT_init])
    
    aug_states = solve_ttd_aug(
        aug_sT_0, aug_ST_top_0, aug_ST_bot_0, aug_sT_init.T, dt, *args, *delta_args, solver=solver, f=f_aug
    )

    ys, ys_dot = aug_states[:, 0, :], aug_states[:, 1, :]
    return ys, ys_dot


In [20]:
# Without the self-defined JVP rule
# def evolve_jax(sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args):
def evolve_jax(sT_0, ST_top_0):
    return solve_ttd_jax(sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args, solver=solve_step_rk2, f=fvec)

def evolve(sT_0, ST_top_0):
    return solve_ttd(sT_0, ST_top_0, ST_bot_0, sT_init, dt, *args, solver=solve_step_rk2, f=fvec)


In [23]:
# primals2 = (sT_0, ST_top_0)
# tangents2 = (delta_sT_0, delta_ST_top_0)
# eqx.filter_jvp(evolve_jax, primals2, tangents2)
# jax.grad(solve_ttd_jax, argnums=0)(
#     *primals, solver=solve_step_rk2, f=fvec
# )

In [477]:
r1, r2 = eqx.filter_jvp(solve_ttd_jax, primals, tangents, solver=solve_step_rk2, f=fvec)
# r1.shape, r2.shape
r1, r2

(Array([[7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03, 7.8400001e-03, 7.8400001e-03,
         7.8400001e-03, 7.8400001e-03],
        [9.6164675e+00, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02, 1.5999872e-02, 1.5999872e-02,
         1.5999872e-02, 1.5999872e-02],
        [1.5957785e+01, 9.2498512e+00, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02, 2.5721980e-02, 2.5721980e-02,
         2.5721980e-02, 2.5721980e-02],
        [2.2422407e+01, 1.6821121e+01, 8.9008932e+00, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02, 3.8349532e-02, 3.8349532e-02,
         3.8349532e-02, 3.8349532e-02],
        [3.4208199e+01, 2.5603579e+01, 1.7597830e+01, 8.5706577e+00,
         5.5525493e-02, 5.5525493e-02, 5.5525493e-02, 5.5525493e-02,
         5.5525493e-02, 5.5525493e-02]], dtype=float32),
 Array([[  0.11600001,   0.11600001,   0.11600001,   0.11600001,
           0.11600001,   0.1

In [379]:
%timeit eqx.filter_jvp(solve_ttd_jax, primals, tangents, solver=solve_step_rk2, f=fvec)

76.9 ms ± 2.03 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


# When we have both $sT$ and $mT$

In [409]:
def f(sTmT, ST, a, b):
    return a * (b - sTmT) + ST

fvec = jax.vmap(f, in_axes=(0,0,None,None))


In [422]:
def solve_ttd_sTmT(
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, *args, solver, f
):
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    
    def step_τ(states, x):
        # sTmT_init_τ : (1+nm,)
        sTmT_init_τ = x
        sT_init_τ = sTmT_init_τ[0]
    
        # sTmT : (nt, 1+nm)
        # ST_top, ST_bot : (nt+1,)
        sTmT, ST_top, ST_bot = states
    
        # Solve the ODE system
        sTmT_new = solver(
            f, sTmT, ST_top[:-1], ST_bot[1:], dt, *args
        ) # (nt, 1+nm)
        sT_new = sTmT_new[...,0]

        # Update ST
        ST_top_new = ST_bot
        ST_bot_new = jnp.concat([sT_init_τ[None] * dt, ST_bot[1:] + sT_new * dt])
        # jax.debug.print('sT_new: {x}; ST_bot: {y}; args: {z}', x=sT_new, y=ST_bot, z=args)
        
        # Variables as the initial condition to the next step
        sTmT_new_rotate = jnp.concat([sTmT_init_τ[None,...],sTmT_new[:-1]], axis=0)
        
        return (sTmT_new_rotate, ST_top_new, ST_bot_new), sTmT_new
    
    _, sTmTs = jax.lax.scan(step_τ, (sTmT_0, ST_top_0, ST_bot_0), sTmT_init)

    return sTmTs


In [424]:
t0, nt, dt = 0, 10, 0.4

sT_init = jnp.array([10., 15., 20., 30., 50.])
mT_init = jnp.array([[3., 5., 3., 7., 4.]]).T
sTmT_init = jnp.concat([sT_init[...,None], mT_init], axis=1)

nm = mT_init.shape[1]

# Initials at age = 0
sTmT_0 = jnp.zeros([nt, 1+nm])
ST_top_0, ST_bot_0 = jnp.zeros(nt+1), jnp.zeros(nt+1) 

# Arguments
a, b = 0.1, 0.2
args = [a, b]


In [426]:
sTmTs = solve_ttd_sTmT(sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, *args, solver=solve_step_rk2, f=fvec)
sTmTs.shape, sTmTs

((5, 10, 2),
 Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02

## Differentiation without IFT

In [493]:
delta_sTmT_0 = jnp.zeros(sTmT_0.shape)
delta_sTmT_init = jnp.zeros(sTmT_init.shape)
delta_ST_top_0 = jnp.zeros(ST_top_0.size)
delta_ST_bot_0 = jnp.zeros(ST_bot_0.shape)

delta_dt = 0.

delta_a = 1.
delta_b = 2.
delta_args = [delta_a, delta_a]


In [495]:
primals = (sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, *args)
tangents = ((delta_sTmT_0, delta_ST_top_0, delta_ST_bot_0, delta_sTmT_init, delta_dt, *delta_args))
r1, r2 = eqx.filter_jvp(solve_ttd_sTmT, primals, tangents, solver=solve_step_rk2, f=fvec)
# r1.shape, r2.shape
r1, r2

(Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02],
         [

In [430]:
%timeit eqx.filter_jvp(solve_ttd_sTmT, primals, tangents, solver=solve_step_rk2, f=fvec)

139 ms ± 3.46 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Differentiation with IFT

In [522]:
def solve_ttd_sTmT_aug(
    aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init, dt, *args, solver, f
):
    
    def step_τ(states, x):
        # aug_sTmT_init_τ : (2,1+nm)
        aug_sTmT_init_τ = x
        aug_sT_init_τ = aug_sTmT_init_τ[:,0]
    
        # aug_sTmT, aug_ST_top, aug_ST_bot : (2,nt,1+nm), (2,nt+1), (2,nt+1)
        aug_sTmT, aug_ST_top, aug_ST_bot = states
    
        # Solve the ODE system
        aug_sTmT_new = solver(
            f, aug_sTmT, aug_ST_top[:,:-1], aug_ST_bot[:,1:], dt, *args
        ) # (2,nt,1+nm)
        aug_sT_new = aug_sTmT_new[...,0]

        # Update ST
        aug_ST_top_new = aug_ST_bot
        aug_ST_bot_new = jnp.concat([aug_sT_init_τ[:,None] * dt, aug_ST_bot[:,1:] + aug_sT_new * dt], axis=1)
        
        # Variables as the initial condition to the next step
        # shape: (2, nt+1, 1+nm)
        aug_sTmT_new_rotate = jnp.concat([aug_sTmT_init_τ[:,None,:],aug_sTmT_new[:,:-1,:]], axis=1)
        
        return (aug_sTmT_new_rotate, aug_ST_top_new, aug_ST_bot_new), aug_sTmT_new
    
    _, aug_sTmTs = jax.lax.scan(step_τ, (aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0), aug_sTmT_init)

    return aug_sTmTs


In [523]:
# solve_ttd_jax = jax.custom_jvp(solve_ttd, nondiff_argnums=(0,1))
solve_ttd_sTmT_jax = eqx.filter_custom_jvp(solve_ttd_sTmT)

@solve_ttd_sTmT_jax.def_jvp
def solve_ttd_sTmT_jax_jvp(primals, tangents, *, solver, f):
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, *args = primals
    delta_sTmT_0, delta_ST_top_0, delta_ST_bot_0, delta_sTmT_init, delta_dt, *delta_args = tangents
    nargs = len(args)
    
    def f_aug(aug_sTmT, aug_ST, *args_and_delta_args):
        # print(args_and_delta_args)
        # print(aug_sTmT)
        # print(aug_ST)
        primal_sTmT, tangent_sTmT = aug_sTmT[0], aug_sTmT[1]
        primal_ST, tangent_ST = aug_ST[0], aug_ST[1]
        args, delta_args = args_and_delta_args[:nargs], args_and_delta_args[nargs:]
        primal_dot, tangent_dot = eqx.filter_jvp(
            f, (primal_sTmT, primal_ST, *args), (tangent_sTmT, tangent_ST, *delta_args)
        )
        return jnp.stack([primal_dot, tangent_dot])

    aug_sTmT_0 = jnp.stack([sTmT_0, delta_sTmT_0])
    aug_ST_top_0 = jnp.stack([ST_top_0, delta_ST_top_0])
    aug_ST_bot_0 = jnp.stack([ST_bot_0, delta_ST_bot_0])
    aug_sTmT_init = jnp.concat([sTmT_init[:,None,:], delta_sTmT_init[:,None,:]], axis=1)
    
    aug_states = solve_ttd_sTmT_aug(
        aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init, dt, *args, *delta_args, solver=solver, f=f_aug
    )

    ys, ys_dot = aug_states[:, 0, :], aug_states[:, 1, :]
    return ys, ys_dot


In [524]:
r1b, r2b = eqx.filter_jvp(solve_ttd_sTmT_jax, primals, tangents, solver=solve_step_rk2, f=fvec)
# r1.shape, r2.shape
r1b, r2b


(Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02],
         [

In [525]:
%timeit eqx.filter_jvp(solve_ttd_sTmT_jax, primals, tangents, solver=solve_step_rk2, f=fvec)

94.3 ms ± 3.46 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


# When we use pytree (eqx.Module) for $\vec{w}$

In [46]:
class Para(eqx.Module):
    a: jnp.array
    b: jnp.array
    # func: Callable
    
    def __init__(self, a, b):
        self.a, self.b = a, b
        # self.func = lambda x: x

def feqx(sT, ST, para):
    a = para.a
    b = para.b
    return a * (b - sT) + ST
    # return a * (b - sT) + a*ST

feqx_vec = jax.vmap(feqx, in_axes=(0,0,None))


In [50]:
def solve_ttd_sTmT2(
    # sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para, *, solver, f
    f, solver, sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para
):
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    
    def step_τ(states, x):
        # sTmT_init_τ : (1+nm,)
        sTmT_init_τ = x
        sT_init_τ = sTmT_init_τ[0]
    
        # sTmT : (nt, 1+nm)
        # ST_top, ST_bot : (nt+1,)
        sTmT, ST_top, ST_bot = states
    
        # Solve the ODE system
        sTmT_new = solver(
            f, sTmT, ST_top[:-1], ST_bot[1:], dt, *[para]
        ) # (nt, 1+nm)
        sT_new = sTmT_new[...,0]

        # Update ST
        ST_top_new = ST_bot
        ST_bot_new = jnp.concat([sT_init_τ[None] * dt, ST_bot[1:] + sT_new * dt])
        # jax.debug.print('sT_new: {x}; ST_bot: {y}; args: {z}', x=sT_new, y=ST_bot, z=args)
        
        # Variables as the initial condition to the next step
        sTmT_new_rotate = jnp.concat([sTmT_init_τ[None,...],sTmT_new[:-1]], axis=0)
        
        return (sTmT_new_rotate, ST_top_new, ST_bot_new), sTmT_new
    
    _, sTmTs = jax.lax.scan(step_τ, (sTmT_0, ST_top_0, ST_bot_0), sTmT_init)

    return sTmTs


In [51]:
t0, nt, dt = 0, 10, 0.4

sT_init = jnp.array([10., 15., 20., 30., 50.])
mT_init = jnp.array([[3., 5., 3., 7., 4.]]).T
sTmT_init = jnp.concat([sT_init[...,None], mT_init], axis=1)

nm = mT_init.shape[1]

# Initials at age = 0
sTmT_0 = jnp.zeros([nt, 1+nm])
ST_top_0, ST_bot_0 = jnp.zeros(nt+1), jnp.zeros(nt+1) 

# Arguments
a, b = 0.1, 0.2
args = [a, b]

para = Para(a=a, b=b)


In [52]:
# sTmTs = solve_ttd_sTmT2(sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para, solver=solve_step_rk2, f=feqx_vec)
sTmTs = solve_ttd_sTmT2(feqx_vec, solve_step_rk2, sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para)
sTmTs.shape, sTmTs


((5, 10, 2),
 Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02

## Differentiation without IFT

In [53]:
delta_sTmT_0 = jnp.zeros(sTmT_0.shape)
delta_sTmT_init = jnp.zeros(sTmT_init.shape)
delta_ST_top_0 = jnp.zeros(ST_top_0.size)
delta_ST_bot_0 = jnp.zeros(ST_bot_0.shape)

delta_dt = 0.

delta_a = 1.
delta_b = 2.
delta_para = Para(a=delta_a, b=delta_b)
# delta_args = [delta_a, delta_a]


In [54]:
def evolve(sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para):
    return solve_ttd_sTmT2(
        feqx_vec, solve_step_rk2, sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para
    )

In [55]:
primals = (sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para)
tangents = (delta_sTmT_0, delta_ST_top_0, delta_ST_bot_0, delta_sTmT_init, delta_dt, delta_para)
r1, r2 = jax.jvp(evolve, primals, tangents)
# r1, r2 = eqx.filter_jvp(solve_ttd_sTmT2, primals, tangents, solver=solve_step_rk2, f=feqx_vec)
# r1.shape, r2.shape
r1, r2

(Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02],
         [

In [561]:
%timeit eqx.filter_jvp(solve_ttd_sTmT_jax, primals, tangents, solver=solve_step_rk2, f=feqx_vec)

93.5 ms ± 4.37 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Differentiation with IFT

In [28]:
def solve_ttd_sTmT2_aug(
    aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init, dt, aug_para, *, solver, f
):
    
    def step_τ(states, x):
        # aug_sTmT_init_τ : (2,1+nm)
        aug_sTmT_init_τ = x
        aug_sT_init_τ = aug_sTmT_init_τ[:,0]
    
        # aug_sTmT, aug_ST_top, aug_ST_bot : (2,nt,1+nm), (2,nt+1), (2,nt+1)
        aug_sTmT, aug_ST_top, aug_ST_bot = states
    
        # Solve the ODE system
        # TODO: ---
        aug_sTmT_new = solver(
            f, aug_sTmT, aug_ST_top[:,:-1], aug_ST_bot[:,1:], dt, aug_para
        ) # (2,nt,1+nm)
        aug_sT_new = aug_sTmT_new[...,0]

        # Update ST
        aug_ST_top_new = aug_ST_bot
        aug_ST_bot_new = jnp.concat([aug_sT_init_τ[:,None] * dt, aug_ST_bot[:,1:] + aug_sT_new * dt], axis=1)
        
        # Variables as the initial condition to the next step
        # shape: (2, nt+1, 1+nm)
        aug_sTmT_new_rotate = jnp.concat([aug_sTmT_init_τ[:,None,:],aug_sTmT_new[:,:-1,:]], axis=1)
        
        return (aug_sTmT_new_rotate, aug_ST_top_new, aug_ST_bot_new), aug_sTmT_new
    
    _, aug_sTmTs = jax.lax.scan(step_τ, (aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0), aug_sTmT_init)

    return aug_sTmTs


In [24]:
solve_ttd_sTmT2_jax = jax.custom_jvp(solve_ttd_sTmT2, nondiff_argnums=(0,1))

@solve_ttd_sTmT2_jax.defjvp
def solve_ttd_sTmT2_jax_jvp(f, solver, primals, tangents):
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para = primals
    delta_sTmT_0, delta_ST_top_0, delta_ST_bot_0, delta_sTmT_init, delta_dt, delta_para = tangents
    # nargs = len(args)
    
    def f_aug(aug_sTmT, aug_ST, aug_para):
        # print(aug_para)
        # print(aug_sTmT)
        # print(aug_ST)
        primal_sTmT, tangent_sTmT = aug_sTmT[0], aug_sTmT[1]
        primal_ST, tangent_ST = aug_ST[0], aug_ST[1]
        para, delta_para = aug_para[0], aug_para[1]
        primal_dot, tangent_dot = jax.jvp(
            f, (primal_sTmT, primal_ST, para), (tangent_sTmT, tangent_ST, delta_para)
        )
        return jnp.stack([primal_dot, tangent_dot])

    aug_sTmT_0 = jnp.stack([sTmT_0, delta_sTmT_0])
    aug_ST_top_0 = jnp.stack([ST_top_0, delta_ST_top_0])
    aug_ST_bot_0 = jnp.stack([ST_bot_0, delta_ST_bot_0])
    aug_sTmT_init = jnp.concat([sTmT_init[:,None,:], delta_sTmT_init[:,None,:]], axis=1)
    aug_para = [para, delta_para]
    
    aug_states = solve_ttd_sTmT2_aug(
        aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init, dt, aug_para, solver=solver, f=f_aug
    )

    ys, ys_dot = aug_states[:, 0, :], aug_states[:, 1, :]
    return ys, ys_dot


In [26]:
def evolve_jax(sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para):
    return solve_ttd_sTmT2_jax(
        feqx_vec, solve_step_rk2, sTmT_0, ST_top_0, ST_bot_0, sTmT_init, dt, para
    )

In [29]:
r1b, r2b = jax.jvp(evolve_jax, primals, tangents)
# r1.shape, r2.shape
r1b, r2b


(Array([[[7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03],
         [7.8400001e-03, 7.8400001e-03]],
 
        [[9.6164675e+00, 2.8908672e+00],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02],
         [1.5999872e-02, 1.5999872e-02]],
 
        [[1.5957785e+01, 6.3497849e+00],
         [9.2498512e+00, 2.7878945e+00],
         [2.5721980e-02, 2.5721980e-02],
         [2.5721980e-02, 2.5721980e-02],
         [

In [560]:
%timeit eqx.filter_jvp(solve_ttd_sTmT2_jax, primals, tangents, solver=solve_step_rk2, f=feqx_vec)

91.4 ms ± 3.13 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


# Backup

In [340]:
@eqx.filter_custom_jvp
def call(x, y, *, f):
    return f(x, y)

@call.def_jvp
def call_jvp(primals, tangents, *, f):
    x, y = primals
    tx, ty = tangents
    # `y` is not differentiated below, so it has a symbolic zero tangent,
    # represented as a `None`.
    assert ty is None
    primal_out = call(x, y, f=f)
    tangent_out = 2 * tx
    return primal_out, tangent_out

x = jnp.array(2.0)
y = jnp.array(2.0)
pri = (x,y)

dx = jnp.array(1.0)
dy = jnp.array(1.0)
tan = (dx,dy)

fn = lambda a, b: a + b
# This only computes gradients for the first argument `x`.
# eqx.filter_grad(call)(x, y, fn=fn)

eqx.filter_jvp(call, pri, tan, **{"f":fn})

AssertionError: 

In [355]:
@eqx.filter_custom_jvp
def call(x, y, *, fn):
    return fn(x, y)

@call.def_jvp
def call_jvp(primals, tangents, *, fn):
    x, y = primals
    tx, ty = tangents
    # `y` is not differentiated below, so it has a symbolic zero tangent,
    # represented as a `None`.
    assert ty is None
    primal_out = call(x, y, fn=fn)
    tangent_out = 2 * tx
    return primal_out, tangent_out

def call_wrap(x, y):
    return call(x, y, fn=fn)

x = jnp.array(2.0)
y = jnp.array(2.0)
pri = (x,y)

dx = jnp.array(1.0)
dy = jnp.array(1.0)
tan = (dx,dy)

fn = lambda a, b: a + b
# This only computes gradients for the first argument `x`.
eqx.filter_grad(call_wrap)(x, y)

# eqx.filter_jvp(call, pri, tan)

Array(2., dtype=float32, weak_type=True)

In [348]:
@eqx.filter_custom_jvp
def call(x, y, *, fn):
    return fn(x, y)

@call.def_jvp
def call_jvp(primals, tangents, *, fn):
    x, y = primals
    tx, ty = tangents
    # `y` is not differentiated below, so it has a symbolic zero tangent,
    # represented as a `None`.
    assert ty is None
    primal_out = call(x, y, fn=fn)
    tangent_out = 2 * tx
    return primal_out, tangent_out

x = jnp.array(2.0)
y = jnp.array(2.0)
fn = lambda a, b: a + b
# This only computes gradients for the first argument `x`.
eqx.filter_grad(call)(x, y, fn=fn)


Array(2., dtype=float32, weak_type=True)

In [373]:
def call(x, y, *args, fn):
    return fn(x, y)

call(x, y, *args, *delta_args, fn=fn)

Array(4., dtype=float32, weak_type=True)